# Lab de Limpeza - Camada Silver: Clientes

**Comentário:** Descreva aqui as regras de negócio identificadas durante a limpeza manual.

In [1]:
import pandas as pd
from pathlib import Path
import glob

In [2]:
if "__file__" in globals():
    BASE_DIR = Path(__file__).resolve().parents[3]
else:
    BASE_DIR = Path.cwd().resolve().parents[2]

In [3]:
BRONZE_PATH = BASE_DIR / "data" / "bronze" / "clientes"
SILVER_PATH = BASE_DIR / "data" / "silver" / "clientes"

In [4]:
files = glob.glob(str(BRONZE_PATH / "**/*.csv"), recursive=True)

In [5]:
print("BASE_DIR:", BASE_DIR)
print("BRONZE_PATH:", BRONZE_PATH)
print("Arquivos encontrados:", files)

BASE_DIR: /home/elladarte/Documentos/teste-tecnico-uberlandia-refresco/refresco-data-case
BRONZE_PATH: /home/elladarte/Documentos/teste-tecnico-uberlandia-refresco/refresco-data-case/data/bronze/clientes
Arquivos encontrados: ['/home/elladarte/Documentos/teste-tecnico-uberlandia-refresco/refresco-data-case/data/bronze/clientes/ingestion_date=2026-04-27T12:01:49/clientes.csv', '/home/elladarte/Documentos/teste-tecnico-uberlandia-refresco/refresco-data-case/data/bronze/clientes/ingestion_date=2026-04-27T17:45:52/clientes.csv', '/home/elladarte/Documentos/teste-tecnico-uberlandia-refresco/refresco-data-case/data/bronze/clientes/ingestion_date=2026-04-27T17:49:03/clientes.csv']


In [6]:
df = pd.concat(
    [pd.read_csv(file, encoding="utf-8", sep=",") for file in files],
    ignore_index=True
)
df

,id_cliente,nome_cliente,cnpj,canal,regiao,cidade,uf,representante,ativo,_ingested_at,_source_file
0,C0001,Distribuidora União,53.923.217/8673-19,Atacado,Triângulo Mineiro,Araguari,MG,Fernanda Souza,S,2026-04-27 12:01:49.719487,clientes.csv
1,C0002,Atacadão do Zé,60.114.370/9791-25,Food Service,Triângulo Mineiro,Uberlândia,MG,Juliana Costa,S,2026-04-27 12:01:49.719487,clientes.csv
2,C0003,Empório Santa Fé,57.210.791/4830-70,Atacado,Alto Paranaíba,Presidente Olegário,MG,Carlos Mendes,S,2026-04-27 12:01:49.719487,clientes.csv
3,C0004,Padaria da Praça,91.943.575/5951-93,Conveniência,Alto Paranaíba,Patos de Minas,MG,Juliana Costa,S,2026-04-27 12:01:49.719487,clientes.csv
4,C0005,Mercado do João,73.218.199/4846-78,Supermercado,Sul de Minas,Varginha,MG,Fernanda Souza,S,2026-04-27 12:01:49.719487,clientes.csv
...,...,...,...,...,...,...,...,...,...,...,...
355,C0200,Distribuidora União,53.923.217/8673-19,Atacado,Triângulo Mineiro,Araguari,MG,Fernanda Souza,S,2026-04-27 17:49:03.889543,clientes.csv
356,C0201,AtacadÃ£o do ZÃ©,60.114.370/9791-25,Food Service,Triângulo Mineiro,Uberlândia,MG,Juliana Costa,S,2026-04-27 17:49:03.889543,clientes.csv
357,C0202,Empório Santa Fé,57.210.791/4830-70,Atacado,Alto Paranaíba,Presidente Olegário,MG,Carlos Mendes,S,2026-04-27 17:49:03.889543,clientes.csv
358,C0203,Padaria da Praça,91.943.575/5951-93,Conveniência,Alto Paranaíba,Patos de Minas,MG,Juliana Costa,S,2026-04-27 17:49:03.889543,clientes.csv


In [7]:
cols_str = ["nome_cliente", "canal", "regiao", "cidade", "uf", "representante"]

for col in cols_str:
    df[col] = df[col].astype(str).str.strip()

In [8]:
df["_ingested_at"] = pd.to_datetime(df["_ingested_at"], errors="coerce")

In [9]:
df["uf"] = df["uf"].str.upper()

In [10]:
df["ativo"].unique()

array(['S', '1', ' S', 'Sim', 'SIM', 's'], dtype=object)

In [11]:
def normalize_ativo(value):
    if pd.isna(value):
        return None

    v = str(value).strip().lower()

    true_values = {"s", "sim", "1", "true", "t", "yes"}
    false_values = {"n", "nao", "não", "0", "false", "f", "no"}

    if v in true_values:
        return True
    elif v in false_values:
        return False
    else:
        return None 


df["ativo"] = df["ativo"].apply(normalize_ativo)

In [12]:
df = df[df["cnpj"].notna()]
df = df[df["nome_cliente"].notna()]

In [13]:
df

,id_cliente,nome_cliente,cnpj,canal,regiao,cidade,uf,representante,ativo,_ingested_at,_source_file
0,C0001,Distribuidora União,53.923.217/8673-19,Atacado,Triângulo Mineiro,Araguari,MG,Fernanda Souza,True,2026-04-27 12:01:49.719487,clientes.csv
1,C0002,Atacadão do Zé,60.114.370/9791-25,Food Service,Triângulo Mineiro,Uberlândia,MG,Juliana Costa,True,2026-04-27 12:01:49.719487,clientes.csv
2,C0003,Empório Santa Fé,57.210.791/4830-70,Atacado,Alto Paranaíba,Presidente Olegário,MG,Carlos Mendes,True,2026-04-27 12:01:49.719487,clientes.csv
3,C0004,Padaria da Praça,91.943.575/5951-93,Conveniência,Alto Paranaíba,Patos de Minas,MG,Juliana Costa,True,2026-04-27 12:01:49.719487,clientes.csv
4,C0005,Mercado do João,73.218.199/4846-78,Supermercado,Sul de Minas,Varginha,MG,Fernanda Souza,True,2026-04-27 12:01:49.719487,clientes.csv
...,...,...,...,...,...,...,...,...,...,...,...
355,C0200,Distribuidora União,53.923.217/8673-19,Atacado,Triângulo Mineiro,Araguari,MG,Fernanda Souza,True,2026-04-27 17:49:03.889543,clientes.csv
356,C0201,AtacadÃ£o do ZÃ©,60.114.370/9791-25,Food Service,Triângulo Mineiro,Uberlândia,MG,Juliana Costa,True,2026-04-27 17:49:03.889543,clientes.csv
357,C0202,Empório Santa Fé,57.210.791/4830-70,Atacado,Alto Paranaíba,Presidente Olegário,MG,Carlos Mendes,True,2026-04-27 17:49:03.889543,clientes.csv
358,C0203,Padaria da Praça,91.943.575/5951-93,Conveniência,Alto Paranaíba,Patos de Minas,MG,Juliana Costa,True,2026-04-27 17:49:03.889543,clientes.csv


In [14]:
df = df.sort_values("_ingested_at")

In [15]:
df = df.drop_duplicates(
    subset=["cnpj"],
    keep="last"
)

In [16]:
df

,id_cliente,nome_cliente,cnpj,canal,regiao,cidade,uf,representante,ativo,_ingested_at,_source_file
316,C0077,Empório da Maria,50.757.175/8372-69,Food Service,Noroeste de Minas,Paracatu,MG,Tatiane Ferreira,True,2026-04-27 17:49:03.889543,clientes.csv
317,C0078,Atacadão da Praça,33.886.232/8088-74,Supermercado,Alto Paranaíba,Presidente Olegário,MG,Carlos Mendes,True,2026-04-27 17:49:03.889543,clientes.csv
318,C0079,Bar Popular,30.430.826/4694-54,Conveniência,Sul de Minas,Passos,MG,Rafael Lima,True,2026-04-27 17:49:03.889543,clientes.csv
319,C0080,Restaurante Central,80.381.228/5961-88,Varejo Tradicional,Alto Paranaíba,Presidente Olegário,MG,Rafael Lima,True,2026-04-27 17:49:03.889543,clientes.csv
320,C0081,Bar Brasil,31.774.739/6531-82,Conveniência,Sul de Minas,Passos,MG,Carlos Mendes,True,2026-04-27 17:49:03.889543,clientes.csv
...,...,...,...,...,...,...,...,...,...,...,...
272,C0033,Supermercado Central,85.510.636/2288-60,Food Service,Sul de Minas,Pouso Alegre,MG,Paulo Rodrigues,True,2026-04-27 17:49:03.889543,clientes.csv
271,C0032,Conveniência do João,31.920.732/8538-98,Varejo Tradicional,Alto Paranaíba,Patrocínio,MG,Paulo Rodrigues,True,2026-04-27 17:49:03.889543,clientes.csv
358,C0203,Padaria da Praça,91.943.575/5951-93,Conveniência,Alto Paranaíba,Patos de Minas,MG,Juliana Costa,True,2026-04-27 17:49:03.889543,clientes.csv
298,C0059,Lanchonete da Maria,59.994.581/1713-91,Conveniência,Sul de Minas,Poços de Caldas,MG,Juliana Costa,True,2026-04-27 17:49:03.889543,clientes.csv
